# Entrenar el clasificador de notas OSM (Colab, GPU)

Baja las etiquetas del labeler, muestra estadísticas, entrena una línea base
(TF-IDF) y SetFit para **categoría** y **tema**, y mide los resultados.

1. Menú **Entorno de ejecución → Cambiar tipo → GPU (T4)**.
2. Corre las celdas en orden.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Librerías

In [ ]:
!pip install -q setfit==1.1.0 sentence-transformers==3.3.1 transformers==4.45.2 datasets==3.1.0

## 2. Configuración

- `FILTERS`: filtros de la API, por ejemplo `lang=es,en&since=2026-10-30`.
  También `user=Rub21`, `category=spam,personal_data`, `theme=food`,
  `changed=1` (solo donde la persona corrigió a la IA).
- `BASE_MODEL`: `paraphrase-multilingual-mpnet-base-v2` (mejor) o
  `paraphrase-multilingual-MiniLM-L12-v2` (más rápido).
- `TRAIN_LANGS`: idiomas para entrenar; los demás solo se miden.
- `ITERATIONS`: pares de textos por ejemplo que arma SetFit (20 es lo usual).

In [ ]:
FILTERS = ""
LABELS_URL = "https://osm-notes.rub21.com/api/labels.csv"

BASE_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
TRAIN_LANGS = {"es", "en", "und"}
TEST_SIZE = 0.2      # 20% de las notas para medir
MIN_PER_CLASS = 8    # clases con menos etiquetas quedan fuera
ITERATIONS = 20
EPOCHS = 1
BATCH_SIZE = 32
SEED = 42
SKIP_SETFIT = False  # True = solo la línea base (segundos)
OUT = "/content/output"

## 3. Etiquetas

In [ ]:
import urllib.request
import pandas as pd

url = LABELS_URL + (f"?{FILTERS}" if FILTERS else "")
urllib.request.urlretrieve(url, "/content/labels.csv")

labels = pd.read_csv("/content/labels.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
labels = labels.replace({"": None})
labels["note_id"] = labels["note_id"].astype(int)
print(len(labels), "etiquetas de", url)
labels.head()

## 4. Estadísticas de los datos

Antes de entrenar: cuántas etiquetas hay por clase, idioma y plantilla, y
cuánto acierta la IA. Una clase con menos de 8 etiquetas queda fuera del
entrenamiento; con menos de ~20 el modelo la aprende mal.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110

df = labels.copy()
df["template"] = df["template"].fillna("(sin plantilla)")
df["user_text"] = df["user_text"].fillna("")
df["words"] = df["user_text"].str.split().str.len()
df["updated_at"] = pd.to_datetime(df["updated_at"], utc=True, errors="coerce")

print(f"Etiquetas: {len(df)} · notas: {df.note_id.nunique()} · personas: {df.user_name.nunique()}")
print(f"Notas con más de una etiqueta: {(df.note_id.value_counts() > 1).sum()}")

fig, ax = plt.subplots(2, 3, figsize=(20, 10))
def bars(s, a, title, min_line=None):
    s.sort_values().plot.barh(ax=a, title=title)
    for i, v in enumerate(s.sort_values()):
        a.text(v, i, f" {v}", va="center", fontsize=8)
    if min_line:
        a.axvline(min_line, color="red", ls="--", lw=1)
bars(df["category"].value_counts(), ax[0, 0], "Por categoría (rojo = 20)", 20)
bars(df["theme"].fillna("(vacío)").value_counts(), ax[0, 1], "Por tema (rojo = 20)", 20)
bars(df["lang"].value_counts().head(15), ax[0, 2], "Por idioma (top 15)")
bars(df["template"].value_counts(), ax[1, 0], "Por plantilla")
bars(df["user_name"].value_counts().head(15), ax[1, 1], "Por persona")
df["updated_at"].dt.strftime("%Y-%m-%d").value_counts().sort_index().plot.bar(ax=ax[1, 2], title="Etiquetas por día", rot=45)
plt.tight_layout(); plt.show()

In [ ]:
# Categoría x tema: qué temas aparecen en cada categoría.
ct = pd.crosstab(df["category"], df["theme"].fillna("(vacío)"))
fig, a = plt.subplots(figsize=(14, 6))
a.imshow(ct.values, cmap="Greens")
a.set_xticks(range(len(ct.columns)), ct.columns, rotation=45, ha="right")
a.set_yticks(range(len(ct.index)), ct.index)
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        if ct.values[i, j]:
            a.text(j, i, ct.values[i, j], ha="center", va="center", fontsize=8)
a.set_title("Categoría x tema"); plt.tight_layout(); plt.show()

# Largo del texto de la persona (palabras), por categoría.
display(df.groupby("category")["words"].describe()[["count", "mean", "50%", "max"]].round(1))

### ¿Cuánto acierta la IA?

Compara `claude_category` / `claude_theme` con la etiqueta final. Si da 100%,
las páginas se aprobaron sin revisar y el modelo solo aprenderá a copiar a la IA.

In [ ]:
ai = df[df["claude_category"].notna()]
if len(ai):
    ok_cat = (ai["category"] == ai["claude_category"]).mean()
    ok_th = (ai["theme"] == ai["claude_theme"]).mean()
    print(f"Con sugerencia de IA: {len(ai)} · acierto categoría {ok_cat:.1%} · tema {ok_th:.1%}")
    per = ai.assign(ok=ai["category"] == ai["claude_category"]).groupby("claude_category")["ok"] \
            .agg(["count", "mean"]).rename(columns={"count": "n", "mean": "acierto"}).sort_values("acierto")
    display(per.style.format({"acierto": "{:.0%}"}))
    changed = ai[ai["category"] != ai["claude_category"]]
    if len(changed):
        print("Qué corrigió la persona (fila = IA, columna = persona):")
        display(pd.crosstab(changed["claude_category"], changed["category"]))
else:
    print("No hay sugerencias de IA en estas etiquetas.")

## 5. Código del entrenamiento

1. Una etiqueta por nota: si varias personas la etiquetaron, gana la mayoría.
2. Entrada del modelo: `[template] [poi_type] user_text`.
3. 80% entrenamiento / 20% prueba, estratificado por clase. Los idiomas fuera
   de `TRAIN_LANGS` van enteros a prueba (solo se miden).
4. Línea base (TF-IDF + regresión logística) y SetFit.
5. Guarda en `OUT/run_<fecha>_<target>/`: `report.md`, `metrics.json`,
   `errors.csv`, `predictions.csv` y `model/`.

In [ ]:
import datetime as dt
import json
import os
from collections import Counter

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split


def model_input(r) -> str:
    """Short template tags + the user's words."""
    tags = "".join(f"[{v}] " for v in (r.template, r.poi_type) if isinstance(v, str) and v)
    return (tags + " ".join(str(r.user_text or "").split())).strip()


def prepare(labels: pd.DataFrame, target: str) -> pd.DataFrame:
    """One row per note with the majority label and the model input text."""
    df = labels[labels[target].notna()]
    if target == "category":
        df = df[df[target] != "unsure"]  # "I don't know" teaches nothing
    if df.empty:
        return df

    def vote(g: pd.DataFrame) -> pd.Series:
        label, n = Counter(g[target]).most_common(1)[0]
        first = g.iloc[0]
        return pd.Series({
            "label": label, "votes": len(g), "agreement": n / len(g),
            "claude": first[f"claude_{target}"], "lang": first["lang"],
            "template": first["template"], "poi_type": first["poi_type"],
            "user_text": first["user_text"]})

    out = df.groupby("note_id").apply(vote, include_groups=False).reset_index()
    out["text"] = [model_input(r) for r in out.itertuples(index=False)]
    return out


def split(df: pd.DataFrame):
    measure_only = df[~df["lang"].isin(TRAIN_LANGS)]
    pool = df[df["lang"].isin(TRAIN_LANGS)]
    counts = pool["label"].value_counts()
    small = sorted(counts[counts < MIN_PER_CLASS].index)
    pool = pool[~pool["label"].isin(small)]
    measure_only = measure_only[~measure_only["label"].isin(small)]
    train, test = train_test_split(pool, test_size=TEST_SIZE, random_state=SEED,
                                   stratify=pool["label"])
    test = pd.concat([test.assign(split="test"), measure_only.assign(split="measure_only")])
    return train, test, small


def baseline(train: pd.DataFrame, test: pd.DataFrame) -> np.ndarray:
    """TF-IDF on character n-grams + logistic regression.
    If SetFit does not beat this, the problem is in the data."""
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)
    clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)
    clf.fit(vec.fit_transform(train["text"]), train["label"])
    return clf.predict(vec.transform(test["text"]))


def train_setfit(train: pd.DataFrame, test: pd.DataFrame, model_dir: str) -> np.ndarray:
    from datasets import Dataset
    from setfit import SetFitModel, Trainer, TrainingArguments

    model = SetFitModel.from_pretrained(BASE_MODEL, labels=sorted(train["label"].unique()))
    trainer = Trainer(
        model=model,
        args=TrainingArguments(batch_size=BATCH_SIZE, num_epochs=EPOCHS,
                               num_iterations=ITERATIONS, seed=SEED, report_to="none"),
        train_dataset=Dataset.from_pandas(train[["text", "label"]].reset_index(drop=True)),
    )
    trainer.train()
    model.save_pretrained(model_dir)
    return np.array(model.predict(test["text"].tolist()))


def scores(y_true, y_pred) -> dict:
    return {"n": int(len(y_true)),
            "accuracy": round(float(accuracy_score(y_true, y_pred)), 3),
            "macro_f1": round(float(f1_score(y_true, y_pred, average="macro", zero_division=0)), 3)}


def by_lang(test: pd.DataFrame, col: str) -> pd.DataFrame:
    rows = [{"lang": lang, "split": g["split"].iloc[0], **scores(g["label"], g[col])}
            for lang, g in test.groupby("lang")]
    return pd.DataFrame(rows).sort_values("n", ascending=False)


def md_table(df: pd.DataFrame) -> str:
    lines = ["| " + " | ".join(map(str, df.columns)) + " |", "|" + "---|" * len(df.columns)]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)


def train_target(target: str) -> str:
    """Train and evaluate one target ("category" or "theme"). Returns the run folder."""
    stamp = dt.datetime.now().strftime("%Y%m%d_%H%M")
    run_dir = os.path.join(OUT, f"run_{stamp}_{target}")
    os.makedirs(run_dir, exist_ok=True)

    df = prepare(labels, target)
    print(f"== {target}: {len(df)} notas etiquetadas")
    if len(df) < 30 or df["label"].nunique() < 2:
        raise SystemExit("Faltan etiquetas (mínimo 30 notas y 2 clases).")

    train, test, small = split(df)
    t_count = int((test["split"] == "test").sum())
    print(f"Train: {len(train)} | test: {t_count} | solo medición: {len(test) - t_count}")
    if small:
        print(f"Fuera (menos de {MIN_PER_CLASS} etiquetas): {', '.join(small)}")

    test = test.copy()
    test["pred_baseline"] = baseline(train, test)
    models = ["baseline"]
    if not SKIP_SETFIT:
        print(f"Entrenando SetFit sobre {BASE_MODEL} ...", flush=True)
        test["pred_setfit"] = train_setfit(train, test, os.path.join(run_dir, "model"))
        models.append("setfit")
    best = models[-1]
    pred = f"pred_{best}"

    t = test[test["split"] == "test"]
    summary = {m: scores(t["label"], t[f"pred_{m}"]) for m in models}
    with_ai = t[t["claude"].notna()]
    if len(with_ai):
        summary["ai_suggestion"] = scores(with_ai["label"], with_ai["claude"])

    report = classification_report(t["label"], t[pred], output_dict=True, zero_division=0)
    per_class = pd.DataFrame([
        {"class": k, "precision": round(v["precision"], 2), "recall": round(v["recall"], 2),
         "f1": round(v["f1-score"], 2), "support": int(v["support"])}
        for k, v in report.items() if k not in ("accuracy", "macro avg", "weighted avg")])
    langs = by_lang(test, pred)
    confusion = pd.crosstab(t["label"], t[pred], rownames=["real"], colnames=["predicted"])

    errors = test[test["label"] != test[pred]]
    errors[["note_id", "lang", "split", "label", pred, "claude", "agreement", "text"]] \
        .to_csv(os.path.join(run_dir, "errors.csv"), index=False, encoding="utf-8-sig")
    test.to_csv(os.path.join(run_dir, "predictions.csv"), index=False, encoding="utf-8-sig")

    metrics = {"target": target, "base_model": BASE_MODEL, "date": stamp,
               "train": len(train), "test": len(t), "measure_only": int(len(test) - len(t)),
               "left_out_classes": small, "summary": summary,
               "per_class": per_class.to_dict("records"), "per_lang": langs.to_dict("records"),
               "low_agreement_notes": int((df["agreement"] < 1).sum()), "filters": FILTERS}
    with open(os.path.join(run_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    summary_df = pd.DataFrame([{"model": k, **v} for k, v in summary.items()])
    left_out = f"Left out (fewer than {MIN_PER_CLASS} labels): {', '.join(small)}" if small else ""
    md = f"""# Run {stamp} · {target}

Base model: `{BASE_MODEL}` · filters: `{FILTERS or "none"}`
Train: {len(train)} · test: {len(t)} · measure-only languages: {len(test) - len(t)}
{left_out}

## Summary (test split)

`ai_suggestion` = how often the AI pre-label (claude_{target}) matched the human label.

{md_table(summary_df)}

## Per class ({best})

{md_table(per_class)}

## Per language ({best})

{md_table(langs)}

## Confusion matrix ({best}, rows = real)

{md_table(confusion.reset_index())}

Errors: `errors.csv` ({len(errors)} notes).
"""
    with open(os.path.join(run_dir, "report.md"), "w") as f:
        f.write(md)
    print(f"Guardado en {run_dir}\n")
    return run_dir

## 6. Entrenar

En una T4 son unos minutos por objetivo.

In [ ]:
run_cat = train_target("category")
run_theme = train_target("theme")

## 7. Resultados

In [ ]:
import json
import matplotlib.pyplot as plt

def show(run):
    m = json.load(open(f"{run}/metrics.json"))
    pred = pd.read_csv(f"{run}/predictions.csv")
    best = "pred_setfit" if "pred_setfit" in pred else "pred_baseline"
    test = pred[pred["split"] == "test"]
    print(f"== {m['target']}  ({run})  train={m['train']} test={m['test']}")
    display(pd.DataFrame(m["summary"]).T)

    fig, ax = plt.subplots(1, 3, figsize=(20, 5))
    s = pd.DataFrame(m["summary"]).T["macro_f1"]
    s.plot.bar(ax=ax[0], title="macro-F1 por modelo", ylim=(0, 1), rot=0)

    pc = pd.DataFrame(m["per_class"]).set_index("class")
    pc[["precision", "recall", "f1"]].plot.barh(ax=ax[1], title=f"Por clase ({best[5:]})", xlim=(0, 1))

    cm = pd.crosstab(test["label"], test[best])
    im = ax[2].imshow(cm.values, cmap="Blues")
    ax[2].set_xticks(range(len(cm.columns)), cm.columns, rotation=60, ha="right")
    ax[2].set_yticks(range(len(cm.index)), cm.index)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax[2].text(j, i, cm.values[i, j], ha="center", va="center", fontsize=8)
    ax[2].set_title("Matriz de confusión (filas = real)")
    plt.tight_layout(); plt.show()

    display(pd.DataFrame(m["per_lang"]))
    return run

show(run_cat)
show(run_theme)

### Línea base vs SetFit por clase

Si SetFit no le gana a la línea base en una clase, faltan ejemplos de esa clase
o las etiquetas son inconsistentes.

In [ ]:
from sklearn.metrics import f1_score

def compare(run):
    pred = pd.read_csv(f"{run}/predictions.csv")
    t = pred[pred["split"] == "test"]
    cols = [c for c in ("pred_baseline", "pred_setfit") if c in t]
    classes = sorted(t["label"].unique())
    out = pd.DataFrame({c[5:]: f1_score(t["label"], t[c], labels=classes, average=None, zero_division=0)
                        for c in cols}, index=classes)
    out["support"] = t["label"].value_counts().reindex(classes).values
    if "setfit" in out:
        out["gana"] = (out["setfit"] - out["baseline"]).round(2)
    ax = out[[c[5:] for c in cols]].plot.barh(figsize=(10, 6), xlim=(0, 1), title=f"F1 por clase · {run.split('/')[-1]}")
    plt.tight_layout(); plt.show()
    return out.round(2)

display(compare(run_cat))
display(compare(run_theme))

### ¿Qué tan seguro está el modelo?

Acierto por largo del texto: las notas muy cortas suelen fallar más.

In [ ]:
def by_length(run):
    pred = pd.read_csv(f"{run}/predictions.csv")
    t = pred[pred["split"] == "test"].copy()
    col = "pred_setfit" if "pred_setfit" in t else "pred_baseline"
    t["words"] = t["user_text"].fillna("").str.split().str.len()
    t["largo"] = pd.cut(t["words"], [-1, 0, 3, 10, 30, 10_000], labels=["0", "1-3", "4-10", "11-30", ">30"])
    return t.assign(ok=t["label"] == t[col]).groupby("largo", observed=False)["ok"] \
            .agg(["count", "mean"]).rename(columns={"count": "n", "mean": "acierto"}).round(2)

print("Categoría"); display(by_length(run_cat))
print("Tema"); display(by_length(run_theme))

## 8. Errores (notas mal clasificadas)

In [ ]:
err = pd.read_csv(f"{run_cat}/errors.csv")
pd.set_option("display.max_colwidth", 200)
err.head(50)

## 9. Descargar el modelo

In [ ]:
from google.colab import files
!cd {OUT} && zip -qr /content/models.zip .
files.download("/content/models.zip")